# Building Elevator — Gradio (Colab + GPU 가속)

- **환경**: BuildingElevator (N층 M대 엘리베이터) — 노트북에 내장
- **에이전트**: 랜덤 에이전트 (앱 실행 중 모델/에이전트 보유 설계)
- **가속**: Device 자동 선택 (CUDA > MPS > CPU), 병렬 env
- **제출**: 유저 이름 입력 후 **제출 (10회 평균)** → 리더보드 제출
- **공정한 채점을 위해 제출 관련 코드(리더보드 제출, userId/username/score 전송 등)는 수정하지 마세요.**

**사용법**: 셀 순서대로 실행. Colab에서 **런타임 → 런타임 유형 변경 → GPU** 선택 시 CUDA 표시.

## 1. 라이브러리 설치


In [ ]:
!pip install -q gradio matplotlib gymnasium torch requests


## 2. 환경 정의 (BuildingElevator 내장)

아래 셀을 실행하면 환경 클래스가 노트북에 정의됩니다.


In [ ]:
"""
BuildingElevator-v0: N층 건물, M대 엘리베이터 제어 환경.

기본: 4층, 1대. num_floors, num_elevators 인자로 변경 가능 (예: 6층 2대, 20층 6대).
렌더 없음.

- mode: "training" | "evaluation". 평가 모드에서는 시드가 고정되어 재현 가능.
- submit_score: True이면 evaluation 모드에서 에피소드 종료(truncated) 시 리더보드 제출.
- user_id: 제출 시 사용 (미지정 시 machine_id 사용).
"""

from __future__ import annotations

import uuid

import gymnasium as gym
from gymnasium import spaces
import numpy as np

LEADERBOARD_URL = "https://us-central1-leaderboard-b29d3.cloudfunctions.net/submitScore"


def get_mac_address() -> str:
    """MAC 기반 식별자 반환 (제출 시 user_id 기본값으로 사용)."""
    mac = ":".join(
        [
            "{:02x}".format((uuid.getnode() >> elements) & 0xFF)
            for elements in range(0, 2 * 6, 2)
        ][::-1]
    )
    return mac


ACTION_STAY = 0
ACTION_MOVE_UP = 1
ACTION_MOVE_DOWN = 2
ACTION_OPEN_DOOR = 3
ACTION_CLOSE_DOOR = 4

NUM_FLOORS_DEFAULT = 4
NUM_ELEVATORS_DEFAULT = 1
# 예: 6층 2대 → num_floors=6, num_elevators=2 / 20층 6대 → num_floors=20, num_elevators=6
CAPACITY = 10
DEFAULT_MAX_STEPS = 1000

REWARD_PER_STEP_PER_WAITING = -0.1
REWARD_PER_ARRIVAL = 10.0
PENALTY_DOOR_OPEN_MOVE = -10.0
PENALTY_BOUNDARY = -1.0
PENALTY_EMPTY_FLOOR_OPEN = -0.5

# 평가 모드에서 시드 미지정 시 사용 (재현 가능한 평가용)
EVAL_DEFAULT_SEED = 0xBEEF  # 48879


class BuildingElevatorEnv(gym.Env):
    """N층 건물, M대 엘리베이터 제어 환경. 기본 4층 1대, num_floors/num_elevators로 변경 가능."""

    metadata = {"render_modes": []}
    _run_id: str | None = None

    def __init__(
        self,
        num_floors: int = NUM_FLOORS_DEFAULT,
        num_elevators: int = NUM_ELEVATORS_DEFAULT,
        max_steps: int = DEFAULT_MAX_STEPS,
        passenger_spawn_rate: float = 0.02,
        use_single_action: bool = True,
        mode: str = "training",
        submit_score: bool = False,
        user_id: str | None = None,
        reward_per_step_per_waiting: float = REWARD_PER_STEP_PER_WAITING,
        reward_per_arrival: float = REWARD_PER_ARRIVAL,
        penalty_door_open_move: float = PENALTY_DOOR_OPEN_MOVE,
        penalty_boundary: float = PENALTY_BOUNDARY,
        penalty_empty_floor_open: float = PENALTY_EMPTY_FLOOR_OPEN,
        render_mode: str | None = None,
        seed: int | None = None,
    ):
        super().__init__()
        if BuildingElevatorEnv._run_id is None:
            BuildingElevatorEnv._run_id = str(uuid.uuid4())
        self.run_id = BuildingElevatorEnv._run_id
        self.machine_id = str(uuid.getnode())

        self.num_floors = num_floors
        self.num_elevators = num_elevators
        self.max_steps = max_steps
        self.passenger_spawn_rate = passenger_spawn_rate
        self.use_single_action = use_single_action
        self.mode = mode
        self.submit_score = submit_score
        self._user_id = user_id if user_id is not None else self.machine_id
        self._reward_per_step_per_waiting = reward_per_step_per_waiting
        self._reward_per_arrival = reward_per_arrival
        self._penalty_door_open_move = penalty_door_open_move
        self._penalty_boundary = penalty_boundary
        self._penalty_empty_floor_open = penalty_empty_floor_open
        self.render_mode = render_mode

        nf, ne = num_floors, num_elevators
        self.observation_space = spaces.Dict({
            "elevator_state": spaces.Box(
                low=np.array([[0, -1, 0, 0]] * ne, dtype=np.float32),
                high=np.array([[nf - 1, 1, 1, CAPACITY]] * ne, dtype=np.float32),
                dtype=np.float32,
            ),
            "hall_up_calls": spaces.MultiBinary(nf),
            "hall_down_calls": spaces.MultiBinary(nf),
            "car_calls": spaces.MultiBinary((ne, nf)),
        })
        if use_single_action:
            self.action_space = spaces.Discrete(ne * 5)
        else:
            self.action_space = spaces.MultiDiscrete([5] * ne)

        self._elevator_floor = np.zeros(ne, dtype=np.int32)
        self._elevator_direction = np.zeros(ne, dtype=np.int32)
        self._door_open = np.zeros(ne, dtype=bool)
        self._load = np.zeros(ne, dtype=np.int32)
        self._car_destinations: list[list[int]] = [[] for _ in range(ne)]
        self._hall_up_calls = np.zeros(nf, dtype=np.uint8)
        self._hall_down_calls = np.zeros(nf, dtype=np.uint8)
        self._car_calls = np.zeros((ne, nf), dtype=np.uint8)
        self._waiting_at_floor: list[list[int]] = [[] for _ in range(nf)]
        self._step_count = 0
        self._max_passengers = 0
        self._episode_reward = 0.0
        self._episode_delivered = 0
        self._rng = np.random.default_rng(seed)

    @property
    def user_id(self) -> str:
        """제출 시 사용할 사용자 식별자 (기본: MAC 기반)."""
        return self._user_id

    def _get_total_waiting(self) -> int:
        return sum(len(w) for w in self._waiting_at_floor) + sum(
            len(self._car_destinations[e]) for e in range(self.num_elevators)
        )

    def _get_current_passengers(self) -> int:
        return self._get_total_waiting()

    def _update_hall_calls_for_floor(self, floor: int) -> None:
        up = any(d > floor for d in self._waiting_at_floor[floor])
        down = any(d < floor for d in self._waiting_at_floor[floor])
        self._hall_up_calls[floor] = 1 if up else 0
        self._hall_down_calls[floor] = 1 if down else 0

    def _spawn_passengers(self) -> None:
        nf = self.num_floors
        for f in range(nf):
            n = self._rng.poisson(self.passenger_spawn_rate)
            for _ in range(n):
                other = [x for x in range(nf) if x != f]
                if not other:
                    continue
                dest = int(self._rng.choice(other))
                self._waiting_at_floor[f].append(dest)
        for f in range(nf):
            self._update_hall_calls_for_floor(f)

    def _build_obs(self) -> dict:
        ne, nf = self.num_elevators, self.num_floors
        elevator_state = np.zeros((ne, 4), dtype=np.float32)
        for e in range(ne):
            elevator_state[e] = (
                float(self._elevator_floor[e]),
                float(self._elevator_direction[e]),
                float(1 if self._door_open[e] else 0),
                float(self._load[e]),
            )
        return {
            "elevator_state": elevator_state,
            "hall_up_calls": self._hall_up_calls.copy(),
            "hall_down_calls": self._hall_down_calls.copy(),
            "car_calls": self._car_calls.copy(),
        }

    def reset(self, seed: int | None = None, options: dict | None = None) -> tuple[dict, dict]:
        if self.mode == "evaluation":
            seed = EVAL_DEFAULT_SEED
        super().reset(seed=seed)
        if seed is not None:
            self._rng = np.random.default_rng(seed)

        ne, nf = self.num_elevators, self.num_floors
        self._elevator_floor = np.zeros(ne, dtype=np.int32)
        self._elevator_direction = np.zeros(ne, dtype=np.int32)
        self._door_open = np.zeros(ne, dtype=bool)
        self._load = np.zeros(ne, dtype=np.int32)
        self._car_destinations = [[] for _ in range(ne)]
        self._hall_up_calls = np.zeros(nf, dtype=np.uint8)
        self._hall_down_calls = np.zeros(nf, dtype=np.uint8)
        self._car_calls = np.zeros((ne, nf), dtype=np.uint8)
        self._waiting_at_floor = [[] for _ in range(nf)]
        self._step_count = 0
        self._max_passengers = 0
        self._episode_reward = 0.0
        self._episode_delivered = 0
        self._spawn_passengers()
        self._max_passengers = max(self._max_passengers, self._get_current_passengers())
        return self._build_obs(), {
            "step": self._step_count,
            "waiting": self._get_total_waiting(),
            "max_passengers": self._max_passengers,
            "episode_reward": self._episode_reward,
            "episode_delivered": self._episode_delivered,
        }

    def submit_score(self, score: float, passengers: int = 0) -> bool:
        """점수 제출 (리더보드). userId=machine_id. runId는 아직 안 보냄."""
        try:
            import requests
        except ImportError:
            print("Score submission failed: install 'requests' to submit (pip install requests)")
            return False
        data = {
            "userId": self.machine_id,
            "username": "User",
            "score": int(round(score)),
            "passengers": int(passengers),
        }
        try:
            response = requests.post(LEADERBOARD_URL, json=data, timeout=10)
            return response.status_code == 200
        except Exception as e:
            print(f"Score submission failed: {e}")
            return False

    def _submit_score(self, score: float) -> None:
        """리더보드 자동 제출 (evaluation + submit_score 시에만 호출)."""
        self.submit_score(score, int(self._episode_delivered))

    def step(self, action: np.ndarray | list | int) -> tuple[dict, float, bool, bool, dict]:
        ne, nf = self.num_elevators, self.num_floors
        reward = 0.0
        step_delivered = 0
        if self.use_single_action:
            a_flat = int(action)
            a_flat = max(0, min(ne * 5 - 1, a_flat))
            elevator_id = a_flat // 5
            act = a_flat % 5
            actions = np.zeros(ne, dtype=np.int32)
            actions[elevator_id] = act
        else:
            actions = np.atleast_1d(np.asarray(action, dtype=np.int32)).ravel()
            if len(actions) != ne:
                actions = np.resize(actions, ne)

        for e in range(ne):
            a = int(actions[e])
            f = self._elevator_floor[e]

            if a == ACTION_OPEN_DOOR and not self._door_open[e]:
                drop_off = sum(1 for d in self._car_destinations[e] if d == f)
                pick_up = len(self._waiting_at_floor[f])
                if drop_off == 0 and pick_up == 0:
                    reward += self._penalty_empty_floor_open

            if self._door_open[e] and (a == ACTION_MOVE_UP or a == ACTION_MOVE_DOWN):
                reward += self._penalty_door_open_move
            if a == ACTION_MOVE_DOWN and f == 0:
                reward += self._penalty_boundary
            if a == ACTION_MOVE_UP and f == nf - 1:
                reward += self._penalty_boundary

            if a == ACTION_MOVE_UP and not self._door_open[e] and f < nf - 1:
                self._elevator_floor[e] += 1
                self._elevator_direction[e] = 1
            elif a == ACTION_MOVE_DOWN and not self._door_open[e] and f > 0:
                self._elevator_floor[e] -= 1
                self._elevator_direction[e] = -1
            elif a == ACTION_OPEN_DOOR:
                if not self._door_open[e]:
                    self._door_open[e] = True
                    remaining = [d for d in self._car_destinations[e] if d != f]
                    delivered = len(self._car_destinations[e]) - len(remaining)
                    step_delivered += delivered
                    self._car_destinations[e] = remaining
                    self._load[e] = len(self._car_destinations[e])
                    for _ in range(delivered):
                        reward += self._reward_per_arrival
                    self._car_calls[e, f] = 0
                    for d in set(self._car_destinations[e]):
                        self._car_calls[e, d] = 1
                    waiting = self._waiting_at_floor[f]
                    if self._elevator_direction[e] > 0:
                        to_board = [d for d in waiting if d > f]
                    elif self._elevator_direction[e] < 0:
                        to_board = [d for d in waiting if d < f]
                    else:
                        to_board = list(waiting)
                    boarded = []
                    for d in to_board:
                        if self._load[e] >= CAPACITY:
                            break
                        self._car_destinations[e].append(d)
                        self._car_calls[e, d] = 1
                        self._load[e] += 1
                        boarded.append(d)
                    self._waiting_at_floor[f] = [d for d in waiting if d not in boarded]
                    self._update_hall_calls_for_floor(f)
            elif a == ACTION_CLOSE_DOOR:
                self._door_open[e] = False

        waiting_count = self._get_total_waiting()
        reward += self._reward_per_step_per_waiting * waiting_count
        self._spawn_passengers()
        self._step_count += 1
        self._episode_reward += reward
        self._episode_delivered += step_delivered
        self._max_passengers = max(self._max_passengers, self._get_current_passengers())
        truncated = self._step_count >= self.max_steps

        if truncated and self.mode == "evaluation" and self.submit_score:
            self._submit_score(self._episode_reward)

        info = {
            "step": self._step_count,
            "waiting": self._get_total_waiting(),
            "max_passengers": self._max_passengers,
            "episode_reward": self._episode_reward,
            "episode_delivered": self._episode_delivered,
        }
        return self._build_obs(), reward, False, truncated, info

    def render(self) -> None:
        return None

    def close(self) -> None:
        pass


gym.register(
    id="BuildingElevator-v0",
    entry_point="skyscraper_elevators_env.building_elevator_env:BuildingElevatorEnv",
    max_episode_steps=DEFAULT_MAX_STEPS,
    kwargs={
        "num_floors": NUM_FLOORS_DEFAULT,
        "num_elevators": NUM_ELEVATORS_DEFAULT,
        "max_steps": DEFAULT_MAX_STEPS,
        "use_single_action": True,
    },
)



## 3. Gradio 앱 실행

아래 셀을 실행하면 Gradio 인터페이스가 시작됩니다.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 앱 실행 중에는 모델(에이전트)을 보유하도록 설계. (예: _agent에 policy 보관)
_agent = {"type": "random"}

def get_device():
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

def get_device_name(device):
    if device == "cuda":
        return "CUDA (GPU)"
    if device == "mps":
        return "Apple Silicon (MPS)"
    return "CPU"

# 환경·에피소드·스텝은 고정 (4층 1대, 1000 스텝). 학습 iter 수만 조절.
def run_episodes_random(n_episodes, seed, n_envs=1):
    num_floors, num_elevators, max_steps = NUM_FLOORS_DEFAULT, NUM_ELEVATORS_DEFAULT, DEFAULT_MAX_STEPS
    device_str = get_device_name(get_device())
    if n_envs <= 1:
        env = BuildingElevatorEnv(num_floors=num_floors, num_elevators=num_elevators, max_steps=max_steps, mode="training")
        returns, delivered_list, max_waiting_list = [], [], []
        for _ in range(n_episodes):
            obs, info = env.reset(seed=seed)
            ep_rew = 0.0
            while True:
                action = env.action_space.sample()
                obs, reward, term, trunc, info = env.step(action)
                ep_rew += reward
                if term or trunc:
                    break
            returns.append(ep_rew)
            delivered_list.append(info.get("episode_delivered", 0))
            max_waiting_list.append(info.get("max_passengers", 0))
        env.close()
        return returns, delivered_list, max_waiting_list, device_str
    try:
        from gymnasium.vector import SyncVectorEnv
    except ImportError:
        return run_episodes_random(n_episodes, seed, n_envs=1)
    def make_env():
        def _init():
            return BuildingElevatorEnv(num_floors=num_floors, num_elevators=num_elevators, max_steps=max_steps, mode="training")
        return _init
    env = SyncVectorEnv([make_env() for _ in range(n_envs)])
    returns, delivered_list, max_waiting_list = [], [], []
    obs, _ = env.reset(seed=seed)
    ep_rew = np.zeros(n_envs)
    done_count, target = 0, n_episodes
    while done_count < target:
        actions = np.array([env.single_action_space.sample() for _ in range(n_envs)])
        obs, rewards, terms, truncs, infos = env.step(actions)
        ep_rew += rewards
        ed = infos.get("episode_delivered", np.zeros(n_envs, dtype=np.int64))
        mp = infos.get("max_passengers", np.zeros(n_envs, dtype=np.int64))
        if not isinstance(ed, np.ndarray):
            ed = np.full(n_envs, ed if np.isscalar(ed) else 0)
        if not isinstance(mp, np.ndarray):
            mp = np.full(n_envs, mp if np.isscalar(mp) else 0)
        dones = terms | truncs
        for i in range(n_envs):
            if dones[i]:
                done_count += 1
                returns.append(float(ep_rew[i]))
                delivered_list.append(int(ed[i]) if i < len(ed) else 0)
                max_waiting_list.append(int(mp[i]) if i < len(mp) else 0)
                ep_rew[i] = 0.0
                if done_count >= target:
                    break
        if done_count >= target:
            break
    env.close()
    return returns, delivered_list, max_waiting_list, device_str

def build_plot(returns, delivered, max_waiting):
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    x = np.arange(1, len(returns) + 1)
    axes[0].plot(x, returns, "o-", alpha=0.7, markersize=4)
    axes[0].axhline(np.mean(returns), color="red", linestyle="--", label=f"평균 {np.mean(returns):.1f}")
    axes[0].set_ylabel("에피소드 리워드")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(x, delivered, "o-", alpha=0.7, markersize=4)
    axes[1].axhline(np.mean(delivered), color="red", linestyle="--", label=f"평균 {np.mean(delivered):.1f}")
    axes[1].set_ylabel("배달 수")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[2].plot(x, max_waiting, "o-", alpha=0.7, markersize=4)
    axes[2].axhline(np.mean(max_waiting), color="red", linestyle="--", label=f"평균 {np.mean(max_waiting):.1f}")
    axes[2].set_xlabel("에피소드")
    axes[2].set_ylabel("최대 대기 인원")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

def gradio_run(n_iters, seed_val, n_envs):
    seed = None if seed_val < 0 else seed_val
    returns, delivered, max_waiting, device_str = run_episodes_random(n_iters, seed, max(1, n_envs))
    fig = build_plot(returns, delivered, max_waiting)
    summary = f"**Device:** {device_str}\n\n**학습 iter 수:** {len(returns)}\n**평균 리워드:** {np.mean(returns):.2f}\n**평균 배달 수:** {np.mean(delivered):.2f}\n**평균 최대 대기 인원:** {np.mean(max_waiting):.2f}\n**환경:** {NUM_FLOORS_DEFAULT}층, 엘리베이터 {NUM_ELEVATORS_DEFAULT}대, max_steps={DEFAULT_MAX_STEPS}\n**병렬 env 수:** {n_envs}"
    return fig, summary

# ---------- 제출 관련: 공정한 채점을 위해 아래 gradio_submit 및 제출 로직은 수정하지 마세요 ----------
SUBMIT_RUNS = 10
def gradio_submit(username):
    try:
        import requests
    except ImportError:
        return "제출 실패: pip install requests 필요"
    env = BuildingElevatorEnv(num_floors=NUM_FLOORS_DEFAULT, num_elevators=NUM_ELEVATORS_DEFAULT, max_steps=DEFAULT_MAX_STEPS, mode="evaluation", submit_score=False, user_id=(username.strip() or None))
    returns, delivered = [], []
    for _ in range(SUBMIT_RUNS):
        obs, info = env.reset(seed=0xBEEF)
        ep_rew = 0.0
        while True:
            action = env.action_space.sample()
            obs, r, term, trunc, info = env.step(action)
            ep_rew += r
            if term or trunc:
                break
        returns.append(ep_rew)
        delivered.append(info.get("episode_delivered", 0))
    env.close()
    avg_score = float(np.mean(returns))
    avg_passengers = int(round(np.mean(delivered)))
    user_id_val = username.strip() or get_mac_address()  # userId는 반드시 있음
    user_display = username.strip()
    try:
        resp = requests.post(LEADERBOARD_URL, json={"userId": user_id_val, "username": user_display, "score": int(round(avg_score)), "passengers": avg_passengers}, timeout=10)
        return f"제출 완료: {user_display or '(이름 없음)'} (10회 평균 리워드 {avg_score:.1f}, 배달 {avg_passengers})" if resp.status_code == 200 else f"제출 실패: HTTP {resp.status_code}"
    except Exception as e:
        return f"제출 오류: {e}"

import gradio as gr
with gr.Blocks(title="Building Elevator (Random Agent)") as demo:
    gr.Markdown("# Building Elevator — 랜덤 에이전트 (Colab + GPU 가속)")
    gr.Markdown("층/엘리베이터/스텝·에피소드 수는 고정. 학습 iter 수만 조절. Device: CUDA > MPS > CPU.")
    with gr.Row():
        n_iters = gr.Slider(1, 50, value=10, step=1, label="학습 iter 수")
        seed_val = gr.Number(value=-1, label="시드 (-1=랜덤)")
        n_envs = gr.Slider(1, 8, value=1, step=1, label="병렬 env 수 (가속)")
    run_btn = gr.Button("실행")
    with gr.Row():
        plot_out = gr.Plot(label="결과")
        text_out = gr.Markdown(label="요약")
    run_btn.click(fn=gradio_run, inputs=[n_iters, seed_val, n_envs], outputs=[plot_out, text_out])
    gr.Markdown("---\n### 리더보드 제출 (10회 평균)")
    with gr.Row():
        username = gr.Textbox(label="유저 이름", placeholder="제출 시 표시될 이름", value="")
        submit_btn = gr.Button("제출 (10회 평균)")
    submit_out = gr.Textbox(label="제출 결과", interactive=False)
    submit_btn.click(fn=gradio_submit, inputs=[username], outputs=submit_out)
demo.launch(share=False)
